# Ingest and Vectorize Zoning Laws

### Tech:

To extract raw text from pdfs use PyPDF2 or PyMuPDF/ for docs: python-docx

Langchain to handle splitting and chunking logic

ChromaDB local vector store in the directory

Embeddings from OpenAI or another service



### Notes:

Semantic Chunking is preferred over Recursive Character splitter (standard) with overlap of some tokens based on semantics. To do that, you need embedding API and then do the semantic chunking

### Read the orlando zoning files

In [ ]:
import os
from docx import Document
from pinecone import Pinecone
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings


from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Verify API key is loaded
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please check your .env file.")
print("✓ API key loaded successfully")

In [ ]:
orlando_zoning_laws_path = "../data/orlando-zoning-laws"
files = os.listdir(orlando_zoning_laws_path)
for f in files:
    print(f)

In [ ]:
# Get only .docx files
docx_files = [f for f in os.listdir(orlando_zoning_laws_path) if f.endswith('.docx')]

# Dictionary to store text from each document
documents = {}

for filename in docx_files:
    filepath = os.path.join(orlando_zoning_laws_path, filename)
    doc = Document(filepath)
    
    # Extract all paragraphs
    text = "\n".join([para.text for para in doc.paragraphs])
    documents[filename] = text
    
    print(f"✓ Loaded: {filename} ({len(text)} characters)")

print(f"\nTotal documents loaded: {len(documents)}")

### Split and Chunk the orlando zoning laws

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Chunk all documents
all_chunks = []

for filename, text in documents.items():
    chunks = splitter.create_documents([text])
    for i, chunk in enumerate(chunks):
        chunk.metadata = {
            "source": filename,
            "chunk_id": i,
            "total_chunks": len(chunks),
            "char_count": len(chunk.page_content),
            "chapter": filename.split("___")[0].replace("_", " "),
        }
    all_chunks.extend(chunks)
    print(f"✓ {filename}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

In [ ]:
# Check a sample chunk
print(all_chunks[2570].page_content[:500])
print("\n--- Metadata ---")
print(all_chunks[2570].metadata)

### Generate Embeddings using OpenAI and Pinecone as vector store

In [ ]:
# Recommended - explicitly pass API key
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key
)

# Test it
# test_vector = embeddings.embed_query("zoning regulations in Orlando")
# print(f"Vector dimensions: {len(test_vector)}")

In [ ]:
# Create vector store from your chunks
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # saves locally
)

print(f"✓ Vector store created with {vectorstore._collection.count()} documents")